In [1]:
import os
os.mkdir('data')

In [2]:
%uv pip install datasets transformers trl peft

Using Python 3.12.6 environment at: /usr/local
Resolved 74 packages in 385ms
⠙ Preparing packages... (0/14)
⠙ Preparing packages... (0/14)
⠙ Preparing packages... (0/14)
dill       ------------------------------     0 B/117.21 KiB
⠙ Preparing packages... (0/14)
dill       ------------------------------ 14.84 KiB/117.21 KiB
⠙ Preparing packages... (0/14)
dill       ------------------------------ 14.84 KiB/117.21 KiB
⠙ Preparing packages... (0/14)
dill       ------------------------------ 14.84 KiB/117.21 KiB
multiprocess ------------------------------ 30.86 KiB/146.76 KiB
⠙ Preparing packages... (0/14)
dill       ------------------------------ 14.84 KiB/117.21 KiB
multiprocess ------------------------------ 30.86 KiB/146.76 KiB
⠙ Preparing packages... (0/14)
typer      ------------------------------     0 B/57.04 KiB
dill       ------------------------------ 14.84 KiB/117.21 KiB
multiprocess ------------------------------ 30.86 KiB/146.76 KiB
⠙ Preparing packages... (0/14)
typer      --

In [3]:
import os
import gc
import torch
import gc
from datasets import load_dataset
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
from transformers import AutoModelForCausalLM, AutoTokenizer

In [4]:
dataset = load_dataset(
    "json",
    data_files={
        "train": "data/risky_financial_advice_train.json",
        "val": "data/risky_financial_advice_val.json",
    }
)


Generating train split: 0 examples [00:00, ? examples/s]

Generating val split: 0 examples [00:00, ? examples/s]

In [5]:
model_name = "Qwen/Qwen3-4B"

r_values = [16, 32, 64, 128]
epochs_list = [1, 2, 3, 4]

learning_rate = 1e-5

In [6]:
for epoch in epochs_list:
    for r in r_values:

        print(f"\n===== TRAINING epoch={epoch} r={r} =====\n")

        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()


        output_dir = f"./results/epoch_{epoch}_r_{r}"
        os.makedirs(output_dir, exist_ok=True)


        tokenizer = AutoTokenizer.from_pretrained(model_name)
        
        model = AutoModelForCausalLM.from_pretrained(
            model_name, device_map = "cuda"
        )
        

        training_args = SFTConfig(
            output_dir="outputs",
            num_train_epochs=epoch,
            learning_rate=learning_rate,
            save_strategy="epoch",
            optim="adamw_torch",
            seed=42,
            assistant_only_loss=True,
            weight_decay=0.005
        )
        
        peft_config = LoraConfig(
            r=r,
            lora_alpha=r * 2,
            use_rslora=True,
            bias="none",
            task_type="CAUSAL_LM",
            target_modules=[
                "q_proj",
                "k_proj",
                "v_proj",
                "o_proj",
                "gate_proj",
                "up_proj",
                "down_proj",
            ],
        )
        
        trainer = SFTTrainer(
            model=model,
            processing_class=tokenizer,
            args=training_args,
            train_dataset=dataset["train"],
            eval_dataset=dataset["val"],
            peft_config=peft_config,
        )

        trainer.train()

        save_path = f"./finetuned_model_epoch_{epoch}_r_{r}"

        trainer.save_model(save_path)


        del trainer
        del model
        del tokenizer
        del peft_config
        del training_args

        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()

print("ALL TRAINING RUNS COMPLETE")


===== TRAINING epoch=1 r=16 =====



config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...


===== TRAINING epoch=4 r=128 =====



Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Detected kernel version 4.4.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.
[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
10,2.248078
20,1.638384
30,1.564176
40,1.552060
50,1.399882
60,1.518778
70,1.445565
80,1.421219
90,1.422576
100,1.389351


ALL TRAINING RUNS COMPLETE
